# How to mask and when
**CS 182 · SimpleStories · causal language modeling**

Does a model trained with part-of-speech (POS) weighted context masking and masking ratio decay (MRD) achieve the lowest clean validation loss? This notebook tests the first hypothesis in the project proposal. Each model starts from random weights and receives the same training stories, in the same order within each seed, with a single pass over fresh blocks.

| Regime | Which words are sampled? | Masking rate over training |
|---|---|---|
| `clean` | None | 0% |
| `uniform` | Equal probability | 15% |
| `uniform_mrd` | Equal probability | 30% → 0% |
| `pos` | Function-word proxies receive more weight | 15% |
| `pos_mrd` | Function-word proxies receive more weight | 30% → 0% |
| `pos_reverse` | Content words receive more weight | 15% |
| `pos_reverse_mrd` | Content words receive more weight | 30% → 0% |

**Run in Colab.** Select **Runtime → Change runtime type → GPU**, then run all cells. The default research run prepares about 110 million training targets once and trains seven regimes for each of three seeds (21 models) sequentially. Select `smoke` for a short wiring check. Runtime depends on the GPU and CPU tagging speed. Drive storage is enabled for the research preset so a later session can reuse prepared data and resume training. The default keeps four full checkpoints per model (roughly 5–6 GB for the 21 runs, plus the corpus). Change `USE_DRIVE` for session-only storage.

**Primary outcome** is final next-token cross-entropy in nats per token on a fixed, clean validation set. Top-1 accuracy is secondary. Equal average masking rates let the fixed-versus-decay comparisons isolate the timing of masking.

In [ ]:
%pip -q install "transformers==4.57.6" "datasets==4.4.1" "nltk==3.9.2"

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import time
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import datasets
import nltk
import transformers
from datasets import load_dataset
from nltk.tag import PerceptronTagger
from nltk.tokenize import TreebankWordTokenizer
from transformers import AutoConfig, AutoTokenizer, LlamaForCausalLM
from tqdm.auto import tqdm
from IPython.display import display

PRESET = "research"  # @param ["research", "smoke"]
USE_DRIVE = PRESET == "research"
SEEDS = [182, 183, 184]  # Each seed is reused across all regimes.
REGIMES = ["clean", "uniform", "uniform_mrd", "pos", "pos_mrd", "pos_reverse", "pos_reverse_mrd"]
BATCH = 16 if PRESET == "research" else 2
LENGTH = 512 if PRESET == "research" else 64
TARGETS = 110_000_000 if PRESET == "research" else 2_048
VAL_BLOCKS = 512 if PRESET == "research" else 8
STEPS = math.ceil(TARGETS / (BATCH * LENGTH))
TRAIN_BLOCKS = BATCH * STEPS
BASE_RATE = 0.15
CONTENT_WEIGHT, OTHER_WEIGHT = 0.75, 1.5
# Values are (content weight, function-word proxy weight). Reversal swaps the pair.
POS_WEIGHTS = {
    "pos": (CONTENT_WEIGHT, OTHER_WEIGHT),
    "pos_mrd": (CONTENT_WEIGHT, OTHER_WEIGHT),
    "pos_reverse": (OTHER_WEIGHT, CONTENT_WEIGHT),
    "pos_reverse_mrd": (OTHER_WEIGHT, CONTENT_WEIGHT),
}
CHECKPOINT_EVERY = 500  # Additional rolling recovery save, in optimizer updates.
assert CHECKPOINT_EVERY >= 1
PEAK_LR, WEIGHT_DECAY = 5e-4, 0.1
MODEL_ID, MODEL_REV = "SimpleStories/SimpleStories-5M", "47201cd194f932403b44313fa9c4ac267ad05ebf"
DATA_ID, DATA_REV = "SimpleStories/SimpleStories", "e63b8adc3b1a1bdc7cac5b500d150b71346b0628"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if PRESET == "research" and DEVICE.type != "cuda":
    raise RuntimeError("Select a GPU runtime in Colab or choose the smoke preset.")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/CS182_masking" if USE_DRIVE else "/content/CS182_masking")
ROOT.mkdir(parents=True, exist_ok=True)
assert PRESET in {"research", "smoke"} and len(set(SEEDS)) == len(SEEDS) > 0
assert STEPS >= 2 and 0 < min(CONTENT_WEIGHT, OTHER_WEIGHT)
assert 2 * BASE_RATE * max(CONTENT_WEIGHT, OTHER_WEIGHT) / min(CONTENT_WEIGHT, OTHER_WEIGHT) <= 1
print(f"{PRESET} | {DEVICE} | {STEPS:,} updates and {TRAIN_BLOCKS * LENGTH:,} targets per model")
print(f"{len(REGIMES) * len(SEEDS)} runs | outputs in {ROOT}")

## 1 · Prepare one shared corpus

Use the published 5M configuration and WordPiece tokenizer. A seeded, buffered shuffle draws from the original training split. A hash of each whitespace-normalized story assigns it to training or validation before packing. Exact normalized duplicates are skipped. The official test split stays available for a later evaluation.

Stories are joined with EOS and packed into fixed-length blocks. A block can contain several stories, and attention can extend across EOS. Each target position appears once per run. The last selected story in each split may be cut to fill its token budget.

The English tagger labels words before they are split into WordPieces. A token belongs to the word with which it has the largest character overlap. All pieces of a word within a block share a mask draw. The tagger sees the story during preprocessing, so its annotation can use right context. Only the resulting corruption is supplied to the causal model during training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV, use_fast=True)
config = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REV)
config.use_cache = False
assert tokenizer.is_fast and tokenizer.eos_token_id is not None
assert len(tokenizer) <= config.vocab_size and LENGTH <= config.max_position_embeddings
assert config.attention_dropout == 0
EOS = tokenizer.eos_token_id
SPECIAL = set(tokenizer.all_special_ids)
WORD_TOKENIZER = TreebankWordTokenizer()
# High bit marks the start of a word. Low bits: 0 protected, 1 content, 2 other.
START = 4
DATA_SPEC = dict(model=MODEL_REV, dataset=DATA_REV, train=TRAIN_BLOCKS * LENGTH + 1,
                 val=VAL_BLOCKS * LENGTH + 1, shuffle=416, pipeline="treebank-pos-v1",
                 nltk=nltk.__version__, datasets=datasets.__version__,
                 transformers=transformers.__version__)

def fingerprint(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:16]

CACHE = ROOT / ("corpus-" + fingerprint(DATA_SPEC))
CACHE.mkdir(exist_ok=True)

def pos_codes(story, spans, tags, ids, offsets):
    codes = np.full(len(ids), START, dtype=np.uint8)
    cursor, previous = 0, -1
    for i, (token, (left, right)) in enumerate(zip(ids, offsets)):
        while cursor < len(spans) and spans[cursor][1] <= left:
            cursor += 1
        if token in SPECIAL or left >= right:
            previous = -1
            continue
        best, overlap, j = -1, 0, cursor
        while j < len(spans) and spans[j][0] < right:
            shared = min(right, spans[j][1]) - max(left, spans[j][0])
            if shared > overlap:
                best, overlap = j, shared
            j += 1
        if best >= 0 and any(c.isalnum() for c in story[slice(*spans[best])]):
            cls = 1 if tags[best].startswith(("NN", "VB", "JJ", "RB")) else 2
            codes[i] = cls + (START if best != previous else 0)
            previous = best
        else:
            previous = -1
    return codes

In [ ]:
def prepare_corpus():
    complete = CACHE / "complete.json"
    paths = {k: CACHE / (k + ".bin") for k in ("train", "val", "pos")}
    sizes = {"train": 2 * DATA_SPEC["train"], "val": 2 * DATA_SPEC["val"], "pos": DATA_SPEC["train"]}
    if complete.exists() and all(p.exists() and p.stat().st_size == sizes[k] for k, p in paths.items()):
        return json.loads(complete.read_text())
    complete.unlink(missing_ok=True)
    nltk.download("averaged_perceptron_tagger_eng", quiet=True, raise_on_error=True)
    tagger = PerceptronTagger()
    rows = load_dataset(DATA_ID, split="train", revision=DATA_REV, streaming=True)
    rows = rows.select_columns(["story"]).shuffle(seed=DATA_SPEC["shuffle"], buffer_size=2048)
    seen, used = set(), {"train": 0, "val": 0}
    counts = {"train": 0, "val": 0}
    total = DATA_SPEC["train"] + DATA_SPEC["val"]
    from contextlib import ExitStack
    with ExitStack() as stack:
        out = {k: stack.enter_context(p.open("wb", buffering=2**20)) for k, p in paths.items()}
        bar = stack.enter_context(tqdm(total=total, desc="Tokenizing and tagging", unit="tok", unit_scale=True))
        for rows_batch in rows.iter(batch_size=128):
            batch = []
            for story in rows_batch["story"]:
                normalized = " ".join(story.split())
                if not normalized:
                    continue
                key = hashlib.blake2b(normalized.encode(), digest_size=16).digest()
                split = "val" if key[0] < 4 else "train"
                if key not in seen and used[split] < DATA_SPEC[split]:
                    seen.add(key)
                    batch.append((split, story))
            if not batch:
                continue
            encoded = tokenizer([s for _, s in batch], return_offsets_mapping=True,
                                add_special_tokens=False, truncation=False, verbose=False)
            for (split, story), ids, offsets in zip(batch, encoded.input_ids, encoded.offset_mapping):
                n = min(DATA_SPEC[split] - used[split], len(ids) + 1)
                if n == 0:
                    continue
                if split == "train":
                    spans = list(WORD_TOKENIZER.span_tokenize(story))
                    tags = [t for _, t in tagger.tag([story[a:b] for a, b in spans])]
                    codes = pos_codes(story, spans, tags, ids, offsets)
                    out["pos"].write(np.append(codes, np.uint8(START))[:n].tobytes())
                out[split].write(np.asarray(ids + [EOS], dtype="<u2")[:n].tobytes())
                counts[split] += 1
                used[split] += n
                bar.update(n)
            if sum(used.values()) == total:
                break
    if used != {k: DATA_SPEC[k] for k in used}:
        raise RuntimeError(f"Fresh stories exhausted before requested budget: {used}")
    info = {"spec": DATA_SPEC, "stories": counts, "tokens": used}
    tmp = complete.with_suffix(".tmp")
    tmp.write_text(json.dumps(info, indent=2))
    tmp.replace(complete)
    return info

print(prepare_corpus())
# Read each binary file once. Shuffling blocks afterwards does not require random Drive I/O.
train_ids = np.fromfile(CACHE / "train.bin", dtype="<u2")
train_pos = np.fromfile(CACHE / "pos.bin", dtype=np.uint8)
val_ids = np.fromfile(CACHE / "val.bin", dtype="<u2")
assert len(train_ids) == len(train_pos) == TRAIN_BLOCKS * LENGTH + 1
assert len(val_ids) == VAL_BLOCKS * LENGTH + 1 and max(train_ids.max(), val_ids.max()) < config.vocab_size
assert np.isin(train_pos[:10000], [START, 1, 2, START + 1, START + 2]).all()

## 2 · Define and check the intervention

Masking replaces a selected input word's embeddings with a learned vector. The decoder keeps its causal attention pattern and predicts **every original next token**, including words whose own input embeddings are hidden. This tests input corruption. Forgetful Causal Masking (FCM) is a related attention-edge intervention.

The original POS arms assign nouns, verbs, adjectives and adverbs (Penn tag prefixes NN, VB, JJ, RB) weight 0.75. All other eligible lexical words get 1.5 and serve as a proxy for function words. The reversed arms swap the two weights. This preserves the original tagging rule, including its treatment of auxiliary verbs tagged VB as content. For example, in “the dog ran,” the original weighting makes “the” twice as likely to be masked as “dog.” The reverse weighting makes “dog” twice as likely as “the.”

Punctuation, special tokens and unmatched offsets are protected in every arm. In each batch we normalize weights over eligible **token positions**, then use one random draw per word. The expected fraction of eligible positions masked therefore matches the scheduled rate for both uniform and POS masking. A word split across a block boundary receives a separate draw in each block.

In [ ]:
def mask_rate(regime, step, steps):
    if regime == "clean":
        return 0.0
    if regime.endswith("_mrd"):
        return 2 * BASE_RATE * (1 - step / (steps - 1))
    return BASE_RATE

def word_mask(codes, regime, rate, generator):
    # codes is [batch, context]. A block can begin in the middle of a word.
    kind = codes % START
    eligible = kind != 0
    if rate == 0:
        return torch.zeros_like(eligible)
    starts = codes >= START
    positions = torch.arange(codes.size(1), device=codes.device).expand_as(codes)
    first_piece = torch.where(starts, positions, 0).cummax(dim=1).values
    draws = torch.rand(codes.shape, generator=generator, device=codes.device).gather(1, first_piece)
    if regime in POS_WEIGHTS:
        content, other = POS_WEIGHTS[regime]
        weights = torch.where(kind == 1, content, other) * eligible
        probability = rate * weights * (eligible.sum() / weights.sum().clamp_min(1e-12))
    else:
        probability = rate
    return eligible & (draws < probability)

def new_model(seed):
    torch.manual_seed(seed)
    model = LlamaForCausalLM(config)   # Constructs random weights from the published configuration.
    extra = torch.empty(config.hidden_size).normal_(std=config.initializer_range)
    model.register_parameter("context_mask", torch.nn.Parameter(extra))
    return model.to(DEVICE)

def logits_for(model, x, mask=None):
    if mask is None:
        return model(input_ids=x, use_cache=False).logits
    embeddings = model.get_input_embeddings()(x)
    inputs = torch.where(mask.unsqueeze(-1), model.context_mask, embeddings)
    return model(inputs_embeds=inputs, use_cache=False).logits

def batch_at(ids, blocks, codes=None):
    ix = np.asarray(blocks)[:, None] * LENGTH + np.arange(LENGTH)
    x = torch.as_tensor(ids[ix].astype(np.int64), device=DEVICE)
    y = torch.as_tensor(ids[ix + 1].astype(np.int64), device=DEVICE)
    pos = None if codes is None else torch.as_tensor(codes[ix], device=DEVICE)
    return x, y, pos

sample = torch.tensor([[START + 1, 1, START + 2, 2, START]], device=DEVICE).repeat(1024, 1)
probe_rng = torch.Generator(device=DEVICE).manual_seed(182)
for name, (content, other) in POS_WEIGHTS.items():
    probe_rng.manual_seed(182)
    probe = word_mask(sample, name, BASE_RATE, probe_rng)
    assert torch.equal(probe[:, 0], probe[:, 1]) and torch.equal(probe[:, 2], probe[:, 3])
    assert not probe[:, 4].any()
    if content != other:
        assert (probe[:, 0].sum() > probe[:, 2].sum()).item() == (content > other)
    assert abs(probe[:, :4].float().mean().item() - BASE_RATE) < 0.04
assert not word_mask(sample, "clean", 0, probe_rng).any()
for name in ("uniform_mrd", "pos_mrd", "pos_reverse_mrd"):
    assert np.isclose(np.mean([mask_rate(name, t, STEPS) for t in range(STEPS)]), BASE_RATE)
x0, y0, _ = batch_at(train_ids, [0])
assert torch.equal(x0[:, 1:], y0[:, :-1])
probe_model = new_model(0).eval()
with torch.no_grad():
    a = logits_for(probe_model, x0)
    torch.testing.assert_close(a, logits_for(probe_model, x0, torch.zeros_like(x0, dtype=torch.bool)))
    changed = x0.clone()
    changed[:, LENGTH // 2:] = (changed[:, LENGTH // 2:] + 1) % config.vocab_size
    torch.testing.assert_close(a[:, :LENGTH // 2], logits_for(probe_model, changed)[:, :LENGTH // 2],
                               rtol=1e-4, atol=1e-5)
n_params = sum(p.numel() for p in probe_model.parameters())
print(f"{n_params:,} trainable parameters | {TRAIN_BLOCKS * LENGTH / n_params:.2f} training targets per parameter")
del probe_model, a

## 3 · Train and evaluate

Each seed fixes initialization, block order, and the random numbers used for masking. All arms use AdamW, gradient clipping, and the same learning-rate schedule. Validation runs on clean input at fixed updates, including the final update. Every regime and seed retains reloadable checkpoints at **25%, 50%, 75%, and 100%** of training. Each includes model weights, optimizer and scaler state, completed step, mask counts and validation history. Between milestones the notebook keeps the latest recovery checkpoint (every 500 updates by default). Rerun with the same settings and storage to resume automatically.

Checkpoints are stored in `RUNS/<regime>-seed<seed>/step-XXXXXXXX.pt`. The results use a new folder so older metrics-only runs cannot be mistaken for checkpointed runs. Previously prepared corpus data can still be reused.

In [ ]:
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
def precision():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if USE_AMP else nullcontext()

def lr_at(step):
    warm = max(1, round(0.05 * STEPS))
    if step < warm:
        return PEAK_LR * (step + 1) / warm
    progress = (step - warm) / max(1, STEPS - warm - 1)
    return PEAK_LR * (0.1 + 0.9 * (1 + math.cos(math.pi * progress)) / 2)

@torch.inference_mode()
def evaluate(model):
    model.eval()
    total_loss, correct, count = 0., 0, 0
    for i in range(0, VAL_BLOCKS, BATCH):
        x, y, _ = batch_at(val_ids, np.arange(i, min(i + BATCH, VAL_BLOCKS)))
        with precision():
            logits = logits_for(model, x)
        total_loss += F.cross_entropy(logits.float().flatten(0, 1), y.flatten(), reduction="sum").item()
        correct += (logits.argmax(-1) == y).sum().item()
        count += y.numel()
    model.train()
    return {"val_loss": total_loss / count, "val_accuracy": correct / count}

SPEC = dict(corpus=fingerprint(DATA_SPEC), length=LENGTH, batch=BATCH, steps=STEPS, lr=PEAK_LR,
            weight_decay=WEIGHT_DECAY, base_rate=BASE_RATE, pos_weights=POS_WEIGHTS,
            precision=str(AMP_DTYPE) if USE_AMP else "fp32", torch=str(torch.__version__), protocol=2)
RUNS = ROOT / (PRESET + "-" + fingerprint(SPEC))
RUNS.mkdir(exist_ok=True)
(RUNS / "settings.json").write_text(json.dumps({"run": SPEC, "data": DATA_SPEC}, indent=2))
EVAL_AT = {max(1, round(STEPS * f)) for f in (.25, .5, .75, 1.)}
SAVE_EVERY = min(STEPS, CHECKPOINT_EVERY)

def atomic_json(path, obj):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    tmp.replace(path)

def checkpoint_path(run, step):
    return run / f"step-{step:08d}.pt"

def checkpoint_step(path):
    return int(path.stem.split("-")[1])

def load_checkpoint(regime, seed, step=None):
    """Reload an evaluated milestone for inference; defaults to the final model."""
    path = checkpoint_path(RUNS / f"{regime}-seed{seed}", STEPS if step is None else step)
    state = torch.load(path, map_location="cpu", weights_only=True)
    model = new_model(seed)
    model.load_state_dict(state["model"])
    return model.eval(), state["history"]

def fit(regime, seed):
    run = RUNS / f"{regime}-seed{seed}"
    run.mkdir(exist_ok=True)
    final = run / "metrics.json"
    if final.exists() and checkpoint_path(run, STEPS).exists():
        print(f"Reusing completed {regime}, seed {seed}")
        return json.loads(final.read_text())
    model = new_model(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    order = np.random.default_rng(seed).permutation(TRAIN_BLOCKS).reshape(STEPS, BATCH)
    rng = torch.Generator(device=DEVICE)
    history, start = [], 0
    totals = torch.zeros(4, dtype=torch.int64, device=DEVICE)
    saved = sorted(run.glob("step-*.pt"), key=checkpoint_step)
    if saved:
        state = torch.load(saved[-1], map_location="cpu", weights_only=True)
        model.load_state_dict(state["model"])
        opt.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        start, history = state["step"], state["history"]
        totals.copy_(state["totals"])
        del state
    bar = tqdm(range(start, STEPS), initial=start, total=STEPS, desc=f"{regime} / {seed}")
    for t in bar:
        x, y, codes = batch_at(train_ids, order[t], train_pos)
        rate = mask_rate(regime, t, STEPS)
        rng.manual_seed(seed * 1000000007 + t)
        mask = word_mask(codes, regime, rate, rng)
        opt.param_groups[0]["lr"] = lr_at(t)
        # An FP16 overflow retries the same data and mask at a smaller scale.
        for attempt in range(12):
            opt.zero_grad(set_to_none=True)
            with precision():
                logits = logits_for(model, x, mask if rate else None)
                loss = F.cross_entropy(logits.float().flatten(0, 1), y.flatten())
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {regime}, step {t+1}")
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=not scaler.is_enabled())
            previous = scaler.get_scale()
            scaler.step(opt)
            scaler.update()
            del logits, loss
            if scaler.get_scale() >= previous:
                break
        else:
            raise FloatingPointError("Repeated FP16 overflow on the same batch.")
        kind = codes % START
        totals += torch.stack([(kind == 1).sum(), (mask & (kind == 1)).sum(),
                               (kind == 2).sum(), (mask & (kind == 2)).sum()])
        step = t + 1
        if step in EVAL_AT:
            n1, m1, n2, m2 = totals.tolist()
            row = dict(seed=seed, regime=regime, step=step, targets=step * BATCH * LENGTH,
                       scheduled_rate=rate, realized_rate=(m1+m2)/max(1,n1+n2),
                       masked_content=m1/max(1,n1), masked_other=m2/max(1,n2), **evaluate(model))
            history.append(row)
            bar.set_postfix(val_loss=f"{row['val_loss']:.4f}", acc=f"{row['val_accuracy']:.3f}")
        if step % SAVE_EVERY == 0 or step in EVAL_AT:
            path = checkpoint_path(run, step)
            tmp = path.with_suffix(".tmp")
            torch.save(dict(model=model.state_dict(), optimizer=opt.state_dict(), scaler=scaler.state_dict(),
                            step=step, seed=seed, regime=regime, history=history, totals=totals.cpu()), tmp)
            tmp.replace(path)
            # Retire older rolling saves only after the new file is complete.
            for previous_path in saved:
                if checkpoint_step(previous_path) not in EVAL_AT:
                    previous_path.unlink(missing_ok=True)
            saved = [path]
    # A disconnect after the final checkpoint can be recovered without retraining.
    atomic_json(final, history)
    return history

In [ ]:
records = []
for seed in SEEDS:
    for regime in REGIMES:
        records.extend(fit(regime, seed))
history = pd.DataFrame(records)
history.to_csv(RUNS / "validation_history.csv", index=False)
print(f"Saved {len(history)} validation measurements in {RUNS}")

## 4 · Read the result

Every point is the mean across training seeds. Error bars show **±1 sample standard deviation (SD)** across seeds at the same training step on the same held-out stories. This measures variation from training randomness. With one seed the notebook shows the point alone. The default uses three seeds.

The original `pos_mrd` hypothesis predicts negative paired loss differences against its four original comparators. The two new arms also let you compare weighting directions directly. For each seed, subtract function-heavy loss from content-heavy loss under the same schedule. A negative difference favors masking content words more often. The tables report these per-seed differences and their across-seed mean and SD.

In [ ]:
final = history.loc[history.step == STEPS].copy()
scores = final.pivot(index="seed", columns="regime", values="val_loss").reindex(index=SEEDS, columns=REGIMES)
assert not scores.isna().any().any()
ranking = final.groupby("regime").agg(
    mean_val_loss=("val_loss", "mean"), sd_val_loss=("val_loss", "std"),
    mean_top1_accuracy=("val_accuracy", "mean"), sd_top1_accuracy=("val_accuracy", "std"),
    mean_masked_fraction=("realized_rate", "mean"), seeds=("seed", "nunique")
).sort_values("mean_val_loss")
print("Final checkpoints, ordered by mean clean validation loss")
display(ranking.round(5))

paired = scores.drop(columns="pos_mrd").rsub(scores["pos_mrd"], axis="index")
original_controls = ["clean", "uniform", "uniform_mrd", "pos"]
print("Original POS + MRD minus each comparator (nats/token)")
display(paired.round(5))
wins = (paired[original_controls] < 0).all(axis=1).sum()
print(f"Original POS + MRD beat all four original controls in {wins} of {len(scores)} seeds.")

direction = pd.DataFrame({
    "fixed": scores["pos_reverse"] - scores["pos"],
    "decay": scores["pos_reverse_mrd"] - scores["pos_mrd"],
})
print("Content-heavy minus function-heavy loss, paired within seed")
display(direction.round(5))
direction_summary = direction.agg(["mean", "std", "count"]).T
print("Across-seed summary of paired differences")
display(direction_summary.round(5))
if PRESET == "smoke":
    print("Smoke preset: use these values only to verify the training pipeline.")
if len(scores) == 1:
    print("Only one seed is available. Sample SD is undefined and error bars are omitted.")
ranking.to_csv(RUNS / "final_ranking.csv")
paired.to_csv(RUNS / "paired_loss_differences.csv")
direction.to_csv(RUNS / "paired_weight_direction.csv")
direction_summary.to_csv(RUNS / "paired_weight_direction_summary.csv")

curves = history.groupby(["regime", "targets"]).agg(
    loss=("val_loss", "mean"), loss_sd=("val_loss", "std"),
    accuracy=("val_accuracy", "mean"), accuracy_sd=("val_accuracy", "std"),
    seeds=("seed", "nunique"),
)
assert (curves.seeds == len(SEEDS)).all()
curves.to_csv(RUNS / "validation_curves_by_seed_summary.csv")
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), layout="constrained")
for name in REGIMES:
    curve = curves.loc[name].sort_index()
    for ax, metric in zip(axes, ("loss", "accuracy")):
        spread = curve[metric + "_sd"] if len(SEEDS) > 1 else None
        ax.errorbar(curve.index / 1e6, curve[metric], yerr=spread, marker="o", markersize=4,
                    capsize=3, linewidth=1.4, label=name)
axes[0].set_ylabel("Clean validation loss (nats/token)")
axes[1].set_ylabel("Clean validation top-1 accuracy")
for ax in axes:
    ax.set_xlabel("Training targets (millions)")
axes[1].legend(fontsize=8)
uncertainty_label = f"Mean ± 1 sample SD across {len(SEEDS)} seeds" if len(SEEDS) > 1 else "Single seed"
fig.suptitle(uncertainty_label)
fig.savefig(RUNS / "masking_results.png", dpi=160)
plt.show()

rate_cols = ["masked_content", "masked_other"]
by_arm = final.groupby("regime")[rate_cols]
rates = by_arm.mean().reindex(REGIMES)
spreads = by_arm.std().reindex(REGIMES)
fig, audit = plt.subplots(figsize=(9, 4), layout="constrained")
positions = np.arange(len(REGIMES))
for column, offset, label in zip(rate_cols, (-0.19, 0.19), ("Content", "Function-word proxy")):
    audit.bar(positions + offset, rates[column], width=.38, label=label, capsize=3,
              yerr=spreads[column] if len(SEEDS) > 1 else None)
audit.set_xticks(positions, REGIMES, rotation=20, ha="right")
audit.set(ylabel="Fraction of eligible token positions masked", title=uncertainty_label)
audit.legend()
fig.savefig(RUNS / "mask_rate_audit.png", dpi=160)
plt.show()
print(f"Results and checkpoints: {RUNS}")

### Reload a trained model

After running the setup and training definitions, use the following example to reload a checkpoint. Use `step=sorted(EVAL_AT)[0]` for the first milestone, or omit `step` for the final model. The history entry gives the clean validation score recorded at that step.

```python
model, saved_history = load_checkpoint("pos_reverse_mrd", seed=SEEDS[0])
print(saved_history[-1])
```

### Interpretation and sources

The loss comparison tests how POS weighting direction and mask-rate decay affect clean next-token prediction on SimpleStories. The mask-rate chart checks whether the intended intervention took place. Establishing that the model learned longer-range dependencies would require a separate context-ablation evaluation.

- SimpleStories data and model configuration — [English dataset](https://huggingface.co/datasets/SimpleStories/SimpleStories) · [original 5M model](https://huggingface.co/SimpleStories/SimpleStories-5M) · [Finke et al.](https://arxiv.org/abs/2504.09184)
- Background on causal context masking — [Liu et al., Forgetful Causal Masking](https://arxiv.org/abs/2210.13432)
- Background on POS masking and decay — [Yang et al., Learning Better Masking for Better Language Model Pre-training](https://aclanthology.org/2023.acl-long.400/)